In [ ]:
```xml
<VSCode.Cell language="markdown">
# ♿ Accessibility & WCAG Compliance Guide

**Building inclusive, accessible Aria applications that work for everyone**

This guide covers:
- WCAG 2.1 Level AA compliance
- Accessible React components
- Screen reader optimization
- Keyboard navigation patterns
- Color contrast and visual accessibility
- Testing & auditing tools
- Inclusive design principles
- Customer success through accessibility
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. WCAG 2.1 Level AA Compliance Framework

### The Four Pillars (POUR)

**Perceivable** — Content must be perceivable to all users
- Provide text alternatives for images
- Provide captions for video/audio
- Ensure sufficient color contrast (4.5:1 ratio for text)
- Don't rely on color alone to convey information

**Operable** — Interface must be operable by everyone
- All functionality available via keyboard
- Users have enough time to read/interact
- Avoid content that causes seizures
- Support multiple input methods (mouse, keyboard, touch)

**Understandable** — Information must be understandable
- Use clear, simple language
- Consistent navigation
- Predictable interactions
- Help users avoid and correct mistakes

**Robust** — Content must work across technologies
- Valid HTML/CSS
- Compatible with assistive technologies
- Compatible with browsers and devices

### WCAG Checklist - Level AA
```markdown
## Level A (Minimum)
- [ ] Text alternatives for images
- [ ] Color contrast 3:1
- [ ] Keyboard accessible
- [ ] Labels associated with inputs
- [ ] Forms can be navigated by keyboard

## Level AA (Recommended)
- [ ] Color contrast 4.5:1 (text), 3:1 (large text)
- [ ] Meaningful link text (not "click here")
- [ ] Multiple ways to navigate content
- [ ] Clear and simple language
- [ ] Consistent navigation patterns
- [ ] Error identification and suggestions
- [ ] Captioning for pre-recorded video
- [ ] Audio description for video

## Level AAA (Enhanced)
- [ ] Sign language for video
- [ ] Extended audio description
- [ ] Simplified and alternative versions
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Accessible React Components

### Accessible Button Component
```typescript
// components/AccessibleButton.tsx
import React from 'react';
import './AccessibleButton.css';

interface AccessibleButtonProps {
  onClick: () => void;
  disabled?: boolean;
  ariaLabel: string;
  ariaDescribedBy?: string;
  children: React.ReactNode;
  variant?: 'primary' | 'secondary' | 'danger';
}

export const AccessibleButton: React.FC<AccessibleButtonProps> = ({
  onClick,
  disabled = false,
  ariaLabel,
  ariaDescribedBy,
  children,
  variant = 'primary'
}) => {
  return (
    <button
      className={`btn btn-${variant}`}
      onClick={onClick}
      disabled={disabled}
      aria-label={ariaLabel}
      aria-describedby={ariaDescribedBy}
      aria-disabled={disabled}
    >
      {children}
    </button>
  );
};
```

### Accessible Form Component
```typescript
// components/AccessibleForm.tsx
interface FormField {
  id: string;
  label: string;
  type: string;
  required?: boolean;
  error?: string;
  helpText?: string;
}

export const AccessibleFormField: React.FC<FormField> = ({
  id,
  label,
  type,
  required,
  error,
  helpText
}) => {
  const errorId = `${id}-error`;
  const helpId = `${id}-help`;

  return (
    <div className="form-field">
      <label htmlFor={id} className="form-label">
        {label}
        {required && <span aria-label="required">*</span>}
      </label>
      
      <input
        id={id}
        type={type}
        required={required}
        aria-required={required}
        aria-describedby={`${error ? errorId : ''} ${helpText ? helpId : ''}`.trim()}
        aria-invalid={!!error}
        className={`form-input ${error ? 'error' : ''}`}
      />
      
      {error && (
        <div id={errorId} className="error-message" role="alert">
          {error}
        </div>
      )}
      
      {helpText && (
        <div id={helpId} className="help-text">
          {helpText}
        </div>
      )}
    </div>
  );
};
```

### Accessible Modal Dialog
```typescript
// components/AccessibleModal.tsx
export const AccessibleModal: React.FC<{
  isOpen: boolean;
  title: string;
  onClose: () => void;
  children: React.ReactNode;
}> = ({ isOpen, title, onClose, children }) => {
  React.useEffect(() => {
    if (isOpen) {
      // Prevent scrolling when modal is open
      document.body.style.overflow = 'hidden';
      
      // Focus on modal when opened
      const modalElement = document.getElementById('modal-content');
      modalElement?.focus();
    }
    
    return () => {
      document.body.style.overflow = 'auto';
    };
  }, [isOpen]);

  if (!isOpen) return null;

  return (
    <div
      role="dialog"
      aria-modal="true"
      aria-labelledby="modal-title"
      className="modal-overlay"
      onClick={onClose}
    >
      <div
        className="modal-content"
        id="modal-content"
        onClick={(e) => e.stopPropagation()}
        tabIndex={-1}
      >
        <h2 id="modal-title">{title}</h2>
        
        <button
          className="modal-close"
          onClick={onClose}
          aria-label="Close dialog"
        >
          ×
        </button>
        
        {children}
      </div>
    </div>
  );
};
```

### Screen Reader Skip Link
```typescript
// components/SkipLink.tsx
export const SkipLink: React.FC = () => {
  return (
    <a href="#main-content" className="skip-link">
      Skip to main content
    </a>
  );
};

/* CSS */
.skip-link {
  position: absolute;
  top: -40px;
  left: 0;
  background: #000;
  color: white;
  padding: 8px;
  text-decoration: none;
  z-index: 100;
}

.skip-link:focus {
  top: 0;
}
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Keyboard Navigation Patterns

### Focus Management
```typescript
// hooks/useFocusManagement.ts
import { useRef, useEffect } from 'react';

export function useFocusTrap(isActive: boolean) {
  const containerRef = useRef<HTMLDivElement>(null);

  useEffect(() => {
    if (!isActive || !containerRef.current) return;

    const focusableElements = containerRef.current.querySelectorAll(
      'button, [href], input, select, textarea, [tabindex]:not([tabindex="-1"])'
    );

    const firstElement = focusableElements[0] as HTMLElement;
    const lastElement = focusableElements[focusableElements.length - 1] as HTMLElement;

    const handleKeyDown = (e: KeyboardEvent) => {
      if (e.key !== 'Tab') return;

      if (e.shiftKey) {
        if (document.activeElement === firstElement) {
          e.preventDefault();
          lastElement.focus();
        }
      } else {
        if (document.activeElement === lastElement) {
          e.preventDefault();
          firstElement.focus();
        }
      }
    };

    containerRef.current.addEventListener('keydown', handleKeyDown);
    return () => containerRef.current?.removeEventListener('keydown', handleKeyDown);
  }, [isActive]);

  return containerRef;
}
```

### Keyboard Shortcuts Component
```typescript
// hooks/useKeyboardShortcut.ts
export function useKeyboardShortcut(
  keys: string[],
  callback: () => void,
  enabled = true
) {
  useEffect(() => {
    if (!enabled) return;

    const handleKeyPress = (e: KeyboardEvent) => {
      const pressed = [
        e.ctrlKey && 'ctrl',
        e.altKey && 'alt',
        e.shiftKey && 'shift',
        e.key.toLowerCase()
      ].filter(Boolean);

      if (keys.every(k => pressed.includes(k))) {
        e.preventDefault();
        callback();
      }
    };

    window.addEventListener('keydown', handleKeyPress);
    return () => window.removeEventListener('keydown', handleKeyPress);
  }, [keys, callback, enabled]);
}

// Usage
export function SaveButton() {
  const [saved, setSaved] = useState(false);

  useKeyboardShortcut(['ctrl', 's'], () => {
    handleSave();
    setSaved(true);
  });

  return (
    <button
      title="Save (Ctrl+S)"
      aria-label="Save changes (Ctrl+S)"
    >
      Save
    </button>
  );
}
```

### Tab Order Documentation
```html
<!-- Structure for proper tab order -->
<header>
  <!-- Skip link (first in tab order) -->
  <a href="#main-content" class="skip-link">Skip to main content</a>
  
  <!-- Header navigation -->
  <nav>
    <a href="/">Home</a>
    <a href="/about">About</a>
  </nav>
</header>

<main id="main-content" tabindex="-1">
  <!-- Main content receives focus -->
</main>

<footer>
  <!-- Footer navigation -->
  <nav>
    <a href="/privacy">Privacy</a>
    <a href="/terms">Terms</a>
  </nav>
</footer>
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Color Contrast & Visual Accessibility

### Contrast Ratio Calculator
```typescript
// utils/contrastRatio.ts
export function getContrastRatio(foreground: string, background: string): number {
  const fgLum = getRelativeLuminance(foreground);
  const bgLum = getRelativeLuminance(background);

  const lighter = Math.max(fgLum, bgLum);
  const darker = Math.min(fgLum, bgLum);

  return (lighter + 0.05) / (darker + 0.05);
}

function getRelativeLuminance(hexColor: string): number {
  const rgb = hexToRgb(hexColor);
  
  const [r, g, b] = [rgb.r, rgb.g, rgb.b].map(val => {
    val = val / 255;
    return val <= 0.03928
      ? val / 12.92
      : Math.pow((val + 0.055) / 1.055, 2.4);
  });

  return 0.2126 * r + 0.7152 * g + 0.0722 * b;
}

export function isWCAGCompliant(
  ratio: number,
  level: 'AA' | 'AAA' = 'AA',
  isLargeText = false
): boolean {
  if (level === 'AA') {
    return isLargeText ? ratio >= 3 : ratio >= 4.5;
  }
  // AAA
  return isLargeText ? ratio >= 4.5 : ratio >= 7;
}
```

### Accessible Color Palette
```css
/* Color palette with WCAG AA compliance */
:root {
  /* Primary colors */
  --primary-dark: #003f87;      /* 4.5:1 on white */
  --primary-light: #0066cc;     /* 7.5:1 on white */
  
  /* Status colors */
  --success-dark: #055a00;      /* 6.3:1 on white */
  --success-light: #00aa00;     /* 5.2:1 on white */
  
  --error-dark: #b50000;        /* 5.8:1 on white */
  --error-light: #ff0000;       /* 3.9:1 on white - use dark on error-light */
  
  --warning-dark: #664d00;      /* 5.6:1 on white */
  --warning-light: #ffaa00;     /* 4.5:1 on white */
  
  /* Neutral grays - tested for contrast */
  --text-dark: #1a1a1a;         /* 20:1 on white */
  --text-light: #4d4d4d;        /* 7.5:1 on white */
  --bg-light: #ffffff;
  --bg-lighter: #f5f5f5;
}

/* Ensure sufficient contrast in all states */
.button {
  background-color: var(--primary-dark);
  color: white;
  /* Contrast ratio: 15:1 ✅ */
}

.button:hover {
  background-color: #002d63;
  /* Contrast ratio: 16.5:1 ✅ */
}

.button:disabled {
  background-color: #cccccc;
  color: #666666;
  /* Contrast ratio: 4.5:1 ✅ */
}
```

### Avoid Color-Only Communication
```typescript
// BAD: Using color alone
<div style={{ color: error ? 'red' : 'green' }}>
  Status
</div>

// GOOD: Color + icon + text
<div className={`status ${error ? 'error' : 'success'}`}>
  <span aria-hidden="true">{error ? '❌' : '✅'}</span>
  <span>{error ? 'Error' : 'Success'}</span>
</div>
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. Testing & Auditing Tools

### Automated Accessibility Testing
```typescript
// e2e/accessibility.spec.ts
import { test, expect } from '@playwright/test';
import { injectAxe, checkA11y } from 'axe-playwright';

test('homepage should have no accessibility violations', async ({ page }) => {
  await page.goto('http://localhost:3000');
  
  // Inject axe-core
  await injectAxe(page);
  
  // Check for violations
  await checkA11y(page, null, {
    detailedReport: true,
    detailedReportOptions: {
      html: true
    }
  });
});

test('chat form should be keyboard navigable', async ({ page }) => {
  await page.goto('http://localhost:3000/chat');
  
  // Tab through form
  await page.keyboard.press('Tab');
  expect(await page.locator(':focus').evaluate(el => el.getAttribute('aria-label')))
    .toBe('Message input');
  
  await page.keyboard.press('Tab');
  expect(await page.locator(':focus').evaluate(el => el.textContent))
    .toContain('Send');
});
```

### Manual Testing Checklist
```markdown
## Keyboard Navigation Testing
- [ ] All interactive elements reachable via Tab
- [ ] Focus order is logical
- [ ] Focus indicator is visible
- [ ] Escape key closes modals/dropdowns
- [ ] Enter key activates buttons/links
- [ ] Arrow keys work for lists/tabs

## Screen Reader Testing (NVDA/JAWS/VoiceOver)
- [ ] Page title announces correctly
- [ ] Headings announce and allow navigation
- [ ] Links announce purpose clearly
- [ ] Form fields announce labels
- [ ] Error messages announced
- [ ] Loading states announced

## Visual Testing
- [ ] Text is at least 12px (16px recommended)
- [ ] Line height is at least 1.5x
- [ ] Color contrast 4.5:1 for normal text
- [ ] Focus indicators visible
- [ ] No text in images only
- [ ] Icons have text alternatives
```

### Accessibility Audit Command
```bash
# Run automated accessibility scan
npm run audit:a11y

# Run with detailed report
npm run audit:a11y -- --report html

# Run specific tests
npm run audit:a11y -- --tests "keyboard,contrast,aria"

# Run with screenshots
npm run audit:a11y -- --screenshots
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 6. Common Accessibility Issues & Fixes

| Issue | Impact | Fix |
|-------|--------|-----|
| Missing alt text on images | Screen reader users can't understand content | Add descriptive alt text to all images |
| No focus indicator | Keyboard users can't see where they are | Apply `:focus` styles to all interactive elements |
| Low color contrast | Vision-impaired users can't read | Use minimum 4.5:1 contrast ratio |
| Placeholder instead of label | Form fields unclear to screen readers | Use proper `<label>` elements |
| Nested clickable elements | Users can't activate nested interactions | Avoid nesting `<button>` inside `<a>` |
| Missing ARIA landmarks | Screen readers can't navigate structure | Use `<header>`, `<nav>`, `<main>`, `<footer>` |
| Keyboard-only trap | Users can't escape focused element | Implement proper focus management |
| Auto-playing media | Distracting and unexpected for some | Provide pause controls |

## Real Examples from Aria

### Aria Character Command (Accessible)
```html
<div role="main" aria-label="Aria character interactive control">
  <label for="command-input">
    Enter command for Aria:
    <span aria-label="Examples: move left, wave, dance, pickup ball">
      (?)
    </span>
  </label>
  
  <input
    id="command-input"
    type="text"
    placeholder="e.g., move left, wave, dance"
    aria-describedby="command-help"
    aria-label="Command input for Aria character"
  />
  
  <button
    type="submit"
    aria-label="Execute command (Enter or Ctrl+Return)"
    onclick="executeCommand()"
  >
    Execute
  </button>
  
  <div id="command-help" class="help-text">
    Type a command and press Enter. Aria will perform the action.
  </div>
</div>
```

### Chat Interface (Keyboard Accessible)
```typescript
// Full keyboard support for chat
export const AccessibleChat: React.FC = () => {
  const [messages, setMessages] = useState<Message[]>([]);
  const inputRef = useRef<HTMLInputElement>(null);
  const listRef = useRef<HTMLUListElement>(null);

  const handleSend = (text: string) => {
    const newMessage = { text, sender: 'user' };
    setMessages([...messages, newMessage]);
    
    // Announce message to screen readers
    announceToScreenReader(`You said: ${text}`);
  };

  // Ctrl+Enter to send
  const handleKeyDown = (e: React.KeyboardEvent) => {
    if ((e.ctrlKey || e.metaKey) && e.key === 'Enter') {
      handleSend(inputRef.current?.value || '');
      inputRef.current!.value = '';
    }
  };

  return (
    <div role="main" aria-label="Chat interface">
      <ul
        role="log"
        aria-live="polite"
        aria-label="Chat messages"
        ref={listRef}
      >
        {messages.map((msg, i) => (
          <li key={i} role="article">
            <strong>{msg.sender}:</strong> {msg.text}
          </li>
        ))}
      </ul>

      <input
        ref={inputRef}
        type="text"
        placeholder="Type your message"
        onKeyDown={handleKeyDown}
        aria-label="Message input"
        aria-describedby="send-hint"
      />
      
      <p id="send-hint" className="help-text">
        Type your message and press Ctrl+Enter to send
      </p>
    </div>
  );
};
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 7. Accessibility Success Metrics

- ✅ Automated audit score: ≥95%
- ✅ Keyboard navigation: 100% of interactive elements
- ✅ Color contrast: 100% of text at 4.5:1+
- ✅ Screen reader: All content perceivable
- ✅ WCAG 2.1 Level AA compliance: 100%
- ✅ User feedback: Accessibility ratings ≥4.5/5
- ✅ Incident rate: <1% accessibility-related issues
- ✅ Adoption: Accessible features used by 15%+ of users

## Accessibility Impact

**By investing in accessibility:**
- ~15% of population benefits directly (disabilities)
- ~100% benefits indirectly (aging, temporary injuries, poor lighting, etc.)
- SEO improves (semantic HTML, alt text)
- Code quality improves (cleaner, more maintainable)
- Legal compliance (ADA, GDPR, accessibility laws)
- Market expansion (accessible = more users)
- Customer loyalty (inclusive companies win users)
</VSCode.Cell>
```